# Destriping the EI maps

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter

import config

PROJECT_ROOT = Path(config.__file__).resolve().parent
LOCAL        = (PROJECT_ROOT / config.LOCAL_PROCESSED_DIR).resolve()
EI_MAPS_DIR  = LOCAL / "ei_maps"
manifest     = pd.read_csv(LOCAL / "manifest.csv")
DISCLOSURE_SUBJECTS = ["p012", "p019", "p027"]

def destripe_ei(ei, window=100):
    """Robust column-offset subtraction: one offset per column, estimated from the median
    down the column (robust to outliers, background-dominated), high-passed to isolate the
    stripe jitter, subtracted row-wise. Under-corrects skin slightly (see conclusion)."""
    col    = np.median(ei, axis=0)                     # robust column profile (median over rows)
    offset = col - median_filter(col, size=window)     # per-column stripe = high-freq jitter
    return ei - offset[np.newaxis, :]

def streaking(img, mask=None, min_px=20):
    """Streaking metric (01b). If mask given, restrict the column means to masked pixels.
    Measures only the column-MEAN (DC) part of the stripe — it is blind to the row-varying
    value-dependent residual, so trust the eye for that (see conclusion)."""
    if mask is None:
        mu, sample = img.mean(0), img
    else:
        cnt = mask.sum(0)
        mu     = np.where(cnt > min_px, (img * mask).sum(0) / np.maximum(cnt, 1), np.nan)
        sample = img[mask.astype(bool)]
    ref = (mu[:-2] + mu[2:]) / 2
    dev = np.abs(mu[1:-1] - ref)
    rng = np.percentile(sample, 99) - np.percentile(sample, 1)
    return float(np.nanmean(dev) / rng)

print("EI maps present:", EI_MAPS_DIR.exists(), " | images:", len(manifest))

### Visual check — original vs destriped (fixed colour scale)

Original EI · destriped EI · removed stripe (`original − destriped`), fixed colour scale. The removed
panel should be **vertical lines only, no face** — confirming the correction takes the stripe, not the
signal.

In [ ]:
EI_VMIN, EI_VMAX = -20, 60

fig, axes = plt.subplots(len(DISCLOSURE_SUBJECTS), 3, figsize=(16, 5 * len(DISCLOSURE_SUBJECTS)))
for row, s in enumerate(DISCLOSURE_SUBJECTS):
    ei = np.load(EI_MAPS_DIR / f"{s}_neutral_left.npy")
    d  = destripe_ei(ei)
    panels = [(ei, "original EI", "magma", EI_VMIN, EI_VMAX),
              (d,  "destriped EI", "magma", EI_VMIN, EI_VMAX),
              (ei - d, "removed stripe", "coolwarm", -12, 12)]
    for ax, (img, title, cmap, vmn, vmx) in zip(axes[row], panels):
        im = ax.imshow(img, cmap=cmap, vmin=vmn, vmax=vmx)
        ax.set_title(f"{s} — {title}"); ax.axis("off")
        plt.colorbar(im, ax=ax, fraction=0.046)

plt.suptitle("Robust column-median destripe (fixed colour scale)", y=1.005)
plt.tight_layout(); plt.show()

**Next:** `destripe_ei` → `src/ei_computation.py`, batch-write `ei_maps_destriped/`, point the target there.